# Tutorial 04 — Importing Data with bw2io

Companion explainer: **04_importing_data.md**. We generate an Excel foreground
template, run it through the `ExcelImporter` pipeline
(read → strategies → match → statistics → write), and show how to debug
unlinked exchanges. Ecoinvent import is shown as a guarded, optional cell.

In [1]:
from pathlib import Path
import openpyxl
import bw2data as bd
import bw2io as bi
import bw2calc as bc

bd.projects.set_current("bw25-tutorials")
BIOSPHERE = next(d for d in bd.databases if "biosphere" in d.lower())

13:16:20-0400

 [

warning  

] 

Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.

## 1. Generate an Excel foreground template
We write it into the repo's `data/` folder so it is reusable and inspectable.

In [2]:
data_dir = Path.cwd().parent / "data"
data_dir.mkdir(exist_ok=True)
xlsx_path = data_dir / "kettle_import.xlsx"

wb = openpyxl.Workbook()
ws = wb.active
ws.title = "inventory"
# NOTE: each Activity block carries an explicit `type: process` row. Without it
# the importer cannot link the self-referencing `production` edge, and every
# activity shows up unlinked. This is the #1 Excel-import gotcha.
rows = [
    ["Database", "t04_kettle_xl"], [],
    ["Activity", "electricity production, coal"],
    ["unit", "kilowatt hour"],
    ["type", "process"],
    ["Exchanges"],
    ["name", "amount", "unit", "database", "type", "categories"],
    ["electricity production, coal", 1.0, "kilowatt hour", "t04_kettle_xl", "production", ""],
    ["Carbon dioxide, fossil", 0.95, "kilogram", BIOSPHERE, "biosphere", "air"],
    [],
    ["Activity", "steel production"],
    ["unit", "kilogram"],
    ["type", "process"],
    ["Exchanges"],
    ["name", "amount", "unit", "database", "type", "categories"],
    ["steel production", 1.0, "kilogram", "t04_kettle_xl", "production", ""],
    ["electricity production, coal", 2.9, "kilowatt hour", "t04_kettle_xl", "technosphere", ""],
    ["Carbon dioxide, fossil", 1.9, "kilogram", BIOSPHERE, "biosphere", "air"],
    [],
    ["Activity", "kettle assembly"],
    ["unit", "unit"],
    ["type", "process"],
    ["Exchanges"],
    ["name", "amount", "unit", "database", "type", "categories"],
    ["kettle assembly", 1.0, "unit", "t04_kettle_xl", "production", ""],
    ["steel production", 1.2, "kilogram", "t04_kettle_xl", "technosphere", ""],
]
for r in rows:
    ws.append(r)
wb.save(xlsx_path)
print("wrote", xlsx_path.name)

wrote

kettle_import.xlsx

## 2. The importer pipeline
read → apply strategies (normalize) → match → check statistics → write.

In [3]:
if "t04_kettle_xl" in bd.databases:
    del bd.databases["t04_kettle_xl"]

imp = bi.ExcelImporter(str(xlsx_path))
imp.apply_strategies()
imp.match_database(fields=["name"])                          # link internal edges
imp.match_database(BIOSPHERE, fields=["name", "categories"])  # link elem. flows
stats = imp.statistics()
print("statistics (datasets, exchanges, unlinked):", stats)

Extracted 1 worksheets in 0.03 seconds

Applying strategy: csv_restore_tuples

Applying strategy: csv_restore_booleans

Applying strategy: csv_numerize

Applying strategy: csv_drop_unknown

...

Applying strategy: link_iterable_by_fields

Applying strategy: link_iterable_by_fields

Graph statistics for `t04_kettle_xl` importer:
3 graph nodes:
	process: 3
7 graph edges:
	production: 3
	biosphere: 2
	technosphere: 2
7 edges to the following databases:
	t04_kettle_xl: 5
	ecoinvent-3.10-biosphere: 2
0 unique unlinked edges (0 total):



statistics (datasets, exchanges, unlinked):

(3, 7, 0, 0)

## 3. Debugging unlinked exchanges
Only write when unlinked == 0. If not, inspect exactly what failed.

In [4]:
n_datasets, n_exchanges, n_unlinked = stats[0], stats[1], stats[2]
if n_unlinked:
    print("UNLINKED:")
    for u in list(imp.unlinked)[:10]:
        print("   ", u.get("name"), "| type:", u.get("type"))
else:
    imp.write_database()
    print("✅ written")

13:16:23-0400

 [

warning  

] 

Not able to determine geocollections for all datasets. This database is not ready for regionalization.

...

info     

] 

Vacuuming database            

Created database: t04_kettle_xl

✅ written

## 4. Calculate on the imported database

In [5]:
# The importer assigns codes by activity hash, so look up by NAME, not code:
kettle = bd.get_node(database="t04_kettle_xl", name="kettle assembly")
gwp = next(m for m in bd.methods
           if "IPCC 2013" in str(m) and "GWP100" in str(m).replace(" ", "")
           and "no LT" not in str(m) and "SLCF" not in str(m))
lca = bc.LCA({kettle: 1}, method=gwp)
lca.lci(); lca.lcia()
print("imported kettle GWP =", round(lca.score, 4), "kg CO2-eq")

imported kettle GWP =

5.586

kg CO2-eq

## 5. Optional: ready-made free background (USEEIO) — network required
Guarded so headless execution never fails offline.

In [6]:
try:
    import socket
    socket.setdefaulttimeout(5)
    available = list(bi.remote.get_projects().keys())
    print("remote prepared projects available:", available[:8], "...")
    print("(To install USEEIO: "
          "bi.remote.install_project('USEEIO-1.1', 'USEEIO'))")
except Exception as e:
    print("skipped remote listing (offline or slow):", type(e).__name__)

remote prepared projects available:

['ecoinvent-3.8-biosphere', 'ecoinvent-3.9.1-biosphere', 'USEEIO-1.1', 'forwast', 'ecoinvent-3.10-biosphere', 'ecoinvent-3.11-biosphere', 'ecoinvent-3.12-biosphere', 'regionalization-example']

...

(To install USEEIO: bi.remote.install_project('USEEIO-1.1', 'USEEIO'))

## 6. Optional: ecoinvent (license required) — guarded
Runs only if ECOINVENT_USERNAME / ECOINVENT_PASSWORD are set.

In [7]:
import os
if os.environ.get("ECOINVENT_USERNAME") and os.environ.get("ECOINVENT_PASSWORD"):
    print("Credentials found. To import a release:")
    print("  bi.import_ecoinvent_release(version='3.10', system_model='cutoff',")
    print("      username=os.environ['ECOINVENT_USERNAME'],")
    print("      password=os.environ['ECOINVENT_PASSWORD'])")
    # left as documented call — not executed automatically to control downloads
else:
    print("No ecoinvent credentials in environment — skipping (this is expected).")
    print("Everything in this hub runs on free data without them.")

No ecoinvent credentials in environment — skipping (this is expected).

Everything in this hub runs on free data without them.

Next: **05 — impact assessment with bw2calc** (single, multi-method, multi-FU).